# Database Fundamentals — Logins, Databases, Schemas, Tables and Views

This demo walks through the SQL Server object hierarchy from the ground up.  By the end you will have built:

- A **server-level Login** (`DemoUser`)
- A **Database** (`DemoDB`)
- A **database User** mapped to the Login
- A **Schema** (`Sales`)
- A **Table** (`Sales.Customer`)
- A **View** (`Sales.vw_CustomerSummary`)

### SQL Server object hierarchy

```
Instance (SQL Server)
└── Login               ← server-level principal (who can connect)
    └── Database        ← isolated container for all objects
        ├── User        ← database-level principal (mapped to a Login)
        └── Schema      ← namespace grouping related objects
            ├── Table
            └── View
```

### Prerequisites

- SQL Server (Express or Developer edition) running locally
- VS Code open with the **mssql** extension installed
- An active mssql connection to `localhost`

### How to run cells

Click the **▶ Run Cell** button to the left of each SQL cell, or press `Ctrl+Shift+E`
with your cursor inside the cell.
Run cells **in order from top to bottom** — each section builds on the previous one.

> ⚠️ **Before running any cell** make sure your mssql connection targets `localhost`.
> Click the connection status bar item at the bottom of VS Code, or use
> `Ctrl+Shift+P` → **MS SQL: Connect** and select your local SQL Server instance.

---

## Section 1 — Logins (Server Level)

A **Login** is a server-level security principal. It controls who is allowed to
connect to the SQL Server instance at all — before any specific database is involved.

SQL Server supports two login types:

| Type | Description |
|------|-------------|
| **Windows Authentication** | The user authenticates via their Windows / Active Directory account. More secure — no password is stored in SQL Server. Preferred in corporate environments. |
| **SQL Server Authentication** | A username and password stored inside SQL Server. Useful for applications and cross-platform access. |

In this demo we create a **SQL Server Authentication** login for simplicity.

> **↕ Snowflake vs SQL Server — Logins vs Users**
>
> In Snowflake there is no "login" as a separate object — a Snowflake **User** is
> the single account-level principal covering both authentication and object access.
> In SQL Server these are deliberately split: a **Login** handles authentication at
> the server, and a **User** (created in Section 3) handles authorisation inside a
> specific database. One Login can have Users in multiple databases simultaneously.

### Syntax

```sql
CREATE LOGIN [name]
    WITH PASSWORD    = N'...',   -- N prefix = Unicode string literal
         CHECK_POLICY = OFF;     -- skip Windows password-complexity rules (demo only)
```

- `sys.server_principals` — system catalog listing all server-level principals (logins, server roles).
- `type = 'S'` — filters to SQL Server Authentication logins (`'U'` = Windows login, `'R'` = server role).

In [ ]:
-- ─── Section 1: Create a Login (Server Level) ──────────────────────────────
-- Logins are server-level objects — we work in master
USE master;
GO

-- Drop the login if it already exists so this cell is re-runnable.
-- SQL Server has no DROP LOGIN IF EXISTS, so we check the catalog manually.
IF EXISTS (
    SELECT 1 FROM sys.server_principals
    WHERE  name = N'DemoUser' AND type = 'S'
)
    DROP LOGIN [DemoUser];
GO

-- Create a SQL Server Authentication login
CREATE LOGIN [DemoUser]
    WITH PASSWORD    = N'Demo@1234!',
         CHECK_POLICY = OFF;          -- disabled for demo; always enable in production
GO

-- Verify — should return one row for DemoUser
SELECT
    name         AS LoginName,
    type_desc    AS AuthType,
    create_date  AS CreatedOn
FROM sys.server_principals
WHERE name = N'DemoUser';

### What just happened?

A **server-level Login** called `DemoUser` was created with SQL Server Authentication.

At this point `DemoUser` can knock on the door of the SQL Server instance — but cannot
open any database yet. That requires a **database User**, which we create in Section 3
after the database exists.

`GO` is a batch separator understood by the mssql client (not by SQL Server itself).
It marks the end of a T-SQL batch and sends the preceding statements to the server
for execution. `CREATE LOGIN` must be the first statement in its batch — hence the
`GO` after the `DROP` block.

---

## Section 2 — Create a Database

A **Database** is an isolated container for all your data objects: tables, views,
schemas, users, stored procedures, and more. Everything created in the remaining
sections lives inside `DemoDB`.

> **Tip:** Dropping a database removes **everything inside it**. That makes
> `DROP DATABASE IF EXISTS DemoDB` a convenient single-line reset for this entire
> demo — re-run Section 2 and continue downwards to start completely fresh.

### Syntax

```sql
DROP DATABASE IF EXISTS DemoDB;   -- remove if it already exists
CREATE DATABASE DemoDB;           -- create fresh
USE DemoDB;                       -- switch connection context to this database
```

- `USE DemoDB` scopes **all subsequent statements** to `DemoDB` for the rest of this session.
- Without `USE`, statements would target `master` (the default system database)
  or whichever database the connection defaulted to on login.

In [ ]:
-- ─── Section 2: Create a Database ─────────────────────────────────────────
USE master;
GO

-- Drop DemoDB if it already exists — also removes all objects inside it
DROP DATABASE IF EXISTS DemoDB;
GO

CREATE DATABASE DemoDB;
GO

-- Switch context — all statements below now run inside DemoDB
USE DemoDB;
GO

-- Verify
SELECT DB_NAME() AS CurrentDatabase;

### What just happened?

`DemoDB` is now a brand-new, empty database.

`USE DemoDB` means every subsequent SQL statement in this session runs inside `DemoDB`
by default — so when we later write `CREATE SCHEMA Sales`, SQL Server knows it belongs
to `DemoDB` without needing the full three-part name.

Dropping and recreating the database resets the `IDENTITY` seed, removes all rows,
and drops all objects, users, and schemas it contained — the cleanest way to restart
this demo from scratch.

---

## Section 3 — Create a Database User and Map It to the Login

A **Login** grants access to the SQL Server instance.
A **User** grants access to a specific database, and is always created *inside* that database.

The `FOR LOGIN` clause links them: it maps the database-level User to the server-level
Login. One Login can have mapped Users in multiple databases at the same time.

> **↕ Snowflake vs SQL Server — Account-level vs Database-level principals**
>
> In Snowflake a User is an account-level object and roles control access to all
> databases from a single place — there is no per-database user object.
> In SQL Server, access is database-scoped: you must create a User in *each* database
> the Login needs to access, and assign roles or permissions in each one individually.
> More setup, but finer-grained control.

### Syntax

```sql
CREATE USER [name] FOR LOGIN [login_name];   -- create user, map to login
ALTER ROLE  [role_name] ADD MEMBER [name];   -- assign to a built-in database role
```

**Common built-in database roles:**

| Role | Access |
|------|--------|
| `db_owner` | Full control of the database (used here for demo simplicity) |
| `db_datareader` | `SELECT` on all tables and views |
| `db_datawriter` | `INSERT`, `UPDATE`, `DELETE` on all tables |

In production, prefer `db_datareader` / `db_datawriter` or custom roles.
Only assign `db_owner` when the user genuinely needs to manage the database itself.

In [ ]:
-- ─── Section 3: Create a Database User ────────────────────────────────────
USE DemoDB;
GO

-- Drop the user if it already exists
DROP USER IF EXISTS [DemoUser];
GO

-- Create a database user and map it to the server-level DemoUser login
CREATE USER [DemoUser] FOR LOGIN [DemoUser];
GO

-- Assign db_owner for demo simplicity
-- Production: use least-privilege roles (db_datareader, db_datawriter, or custom)
ALTER ROLE [db_owner] ADD MEMBER [DemoUser];
GO

-- Verify: username, auth type, mapped login, and role membership
SELECT
    dp.name         AS UserName,
    dp.type_desc    AS UserType,
    sp.name         AS MappedLogin,
    rp.name         AS RoleName
FROM sys.database_principals     dp
LEFT JOIN sys.server_principals       sp  ON dp.sid               = sp.sid
LEFT JOIN sys.database_role_members  drm  ON dp.principal_id      = drm.member_principal_id
LEFT JOIN sys.database_principals     rp  ON drm.role_principal_id = rp.principal_id
WHERE dp.name = N'DemoUser';

### What just happened?

A **database-level User** called `DemoUser` was created inside `DemoDB` and linked to
the server-level `DemoUser` Login via the `FOR LOGIN` clause.

The access chain is now complete:

```
SQL Server instance (localhost)
└── Login: DemoUser       ← can connect to the server
    └── Database: DemoDB
        └── User: DemoUser (db_owner)  ← can access and fully manage DemoDB
```

---

## Section 4 — Create a Schema

A **Schema** is a namespace inside a database that groups related objects together.
SQL Server creates a default schema called `dbo` for every database.
Teams typically add domain-specific schemas to keep large databases organised — for
example: `Sales`, `Finance`, `HR`, `Reporting`.

Objects in SQL Server are referenced using a three-part name:

```
DatabaseName.SchemaName.ObjectName
      DemoDB.Sales.Customer
```

> **↕ Snowflake vs SQL Server — Schema hierarchy**
>
> Both platforms use schemas as a grouping namespace. The hierarchy differs slightly:
>
> - **Snowflake:** `Account > Database > Schema > Table`
> - **SQL Server:** `Instance > Database > Schema > Table`
>
> In SQL Server there is no "Account" tier — the instance *is* the top level.
> Schema creation syntax is nearly identical on both platforms:
> `CREATE SCHEMA schema_name;` works in both.

### Syntax

```sql
CREATE SCHEMA [schema_name];
```

A schema can only be **dropped when it is empty** (no tables, views, or other
objects inside it). The cell below drops any dependent objects first so the
notebook remains fully re-runnable even if you jump straight to this section.

In [ ]:
-- ─── Section 4: Create a Schema ────────────────────────────────────────────
USE DemoDB;
GO

-- Drop dependent objects first so the schema can be dropped cleanly.
-- A schema cannot be dropped while it still contains any objects.
DROP VIEW  IF EXISTS [Sales].[vw_CustomerSummary];
DROP TABLE IF EXISTS [Sales].[Customer];
DROP SCHEMA IF EXISTS [Sales];
GO

CREATE SCHEMA [Sales];
GO

-- Verify
SELECT name, schema_id
FROM   sys.schemas
WHERE  name = N'Sales';

### What just happened?

The `Sales` schema now exists inside `DemoDB`.
Any object created with the `Sales.` prefix belongs to this schema, keeping it
visually and logically separate from `dbo` objects.

Schemas also simplify permission management — you can grant a role access to every
object in a schema with a single statement rather than granting object by object:

```sql
GRANT SELECT ON SCHEMA::Sales TO [SomeUser];
```

---

## Section 5 — Create a Table

A **Table** stores structured data in rows and columns. Each column has a defined
**data type** and optional **constraints** that enforce data quality at the database level.

Key column properties used here:

| Property | Purpose |
|----------|---------|
| `IDENTITY(1,1)` | Auto-generates a unique integer for each new row. Seed = 1, increment = 1. |
| `PRIMARY KEY` | Uniquely identifies each row. Implies `NOT NULL` and creates a clustered index. |
| `NOT NULL` | The column must always have a value — empty inserts are rejected. |
| `DEFAULT GETDATE()` | Automatically fills the column with the current date if no value is supplied on insert. |

> **↕ Snowflake vs SQL Server — IDENTITY vs AUTOINCREMENT**
>
> Snowflake uses `AUTOINCREMENT` or `IDENTITY` (both supported, same semantics).
> SQL Server uses `IDENTITY(seed, increment)`. Both produce an auto-incrementing
> integer column — only the keyword differs.

> **↕ Snowflake vs SQL Server — DEFAULT date functions**
>
> SQL Server uses `GETDATE()` to return the current date and time.
> Snowflake equivalents: `CURRENT_DATE()` (date only) or `CURRENT_TIMESTAMP()` (with time).
> SQL Server also has `GETUTCDATE()` for UTC and `SYSDATETIMEOFFSET()` for
> timezone-aware values.

### Syntax

```sql
CREATE TABLE [Schema].[TableName]
(
    ColumnName   DataType   [IDENTITY(seed, increment)]   [NULL | NOT NULL]   [DEFAULT expr],
    ...
    CONSTRAINT ConstraintName PRIMARY KEY (ColumnName)
);
```

- `NVARCHAR(n)` stores Unicode text up to `n` characters. Always prefer `NVARCHAR`
  over `VARCHAR` for names and free-text fields — it handles accented letters and
  non-Latin scripts safely.
- `DATE` stores a date without a time component — equivalent to Snowflake `DATE`.
- `INFORMATION_SCHEMA.COLUMNS` is a standard SQL view for inspecting table structure.
  It works in both SQL Server and Snowflake.

In [ ]:
-- ─── Section 5: Create a Table ─────────────────────────────────────────────
USE DemoDB;
GO

DROP TABLE IF EXISTS [Sales].[Customer];
GO

CREATE TABLE [Sales].[Customer]
(
    CustomerID   INT            IDENTITY(1,1)       NOT NULL,
    FirstName    NVARCHAR(50)                       NOT NULL,
    LastName     NVARCHAR(50)                       NOT NULL,
    Email        NVARCHAR(100)                      NOT NULL,
    CreatedDate  DATE           DEFAULT GETDATE()   NOT NULL,

    CONSTRAINT PK_Customer PRIMARY KEY (CustomerID)
);
GO

-- Verify the column definitions
SELECT
    COLUMN_NAME,
    DATA_TYPE,
    CHARACTER_MAXIMUM_LENGTH   AS MaxLength,
    IS_NULLABLE,
    COLUMN_DEFAULT
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'Sales'
  AND TABLE_NAME   = 'Customer'
ORDER BY ORDINAL_POSITION;

### What just happened?

`Sales.Customer` has been created with five columns:

- `CustomerID` — auto-generated by `IDENTITY`. Never inserted manually, guaranteed unique.
- `FirstName` / `LastName` / `Email` — Unicode text, all mandatory.
- `CreatedDate` — defaults to today's date if omitted on insert.

The query against `INFORMATION_SCHEMA.COLUMNS` is a portable way to inspect table
structure — the same query works in Snowflake, giving you a consistent inspection
tool across both platforms.

---

## Section 6 — Insert Data

`INSERT INTO` adds new rows to a table. When a column uses `IDENTITY`, you must
**omit it from the column list** — SQL Server generates the value automatically.
Attempting to insert into an `IDENTITY` column raises an error unless
`SET IDENTITY_INSERT` is explicitly enabled (an advanced scenario not covered here).

### Syntax

```sql
INSERT INTO [Schema].[Table] (Column1, Column2, ...)
VALUES
    (row1_col1, row1_col2, ...),
    (row2_col1, row2_col2, ...);   -- multi-row insert in a single statement
```

- `CustomerID` is omitted — SQL Server fills it in automatically.
- String literals are prefixed with `N` (`N'Alice'`) to store them as Unicode, matching
  the `NVARCHAR` column type and avoiding an implicit conversion.
- Running this cell multiple times inserts duplicate rows with new IDs.
  To get a clean 5-row result, re-run Section 5 first to reset the table.

In [ ]:
-- ─── Section 6: Insert Sample Data ─────────────────────────────────────────
USE DemoDB;
GO

-- CustomerID is generated by IDENTITY — omit it from the column list
INSERT INTO [Sales].[Customer] (FirstName, LastName, Email, CreatedDate)
VALUES
    (N'Alice',   N'Nguyen',    N'alice.nguyen@example.com',    '2024-01-15'),
    (N'Ben',     N'Carter',    N'ben.carter@example.com',      '2024-02-20'),
    (N'Clara',   N'Osei',      N'clara.osei@example.com',      '2024-03-05'),
    (N'David',   N'Martins',   N'david.martins@example.com',   '2024-04-11'),
    (N'Emma',    N'Johansson', N'emma.johansson@example.com',  '2024-05-30');
GO

-- Show all rows
SELECT * FROM [Sales].[Customer];

### What just happened?

Five rows were inserted into `Sales.Customer`.
`CustomerID` was assigned automatically by `IDENTITY` (1 through 5) — you never
specified these values.

All five rows have distinct `CreatedDate` values because we supplied them explicitly.
If you had omitted `CreatedDate` from the `INSERT`, SQL Server would have filled it
with `GETDATE()` — today's date — for every row. This is the `DEFAULT` constraint
at work: useful data without cluttering every `INSERT` statement.

---

## Section 7 — Create a View

A **View** is a saved `SELECT` query stored in the database under a name.
It does not store data — every time you query the view, SQL Server executes the
underlying `SELECT` against the real table(s) and returns a fresh result.

Views are used to:

- **Simplify queries** — hide complex joins or calculations behind a clean, reusable name.
- **Control column exposure** — grant access to a view that omits sensitive columns
  (e.g. salary, PII) while blocking direct table access.
- **Provide a stable interface** — the underlying table structure can evolve without
  breaking upstream queries that use the view.

> **↕ Snowflake vs SQL Server — Views**
>
> Views work **identically** in SQL Server and Snowflake: a saved SELECT query,
> no stored data, re-executed on every query. This is a reliable anchor of familiarity.
>
> Snowflake additionally supports **Secure Views** (hides the definition from
> non-owners) and **Materialized Views** (stores the result for performance).
> SQL Server has **Indexed Views** as the equivalent of Materialized Views, but
> they require extra setup. A plain `CREATE VIEW` behaves the same on both platforms.

### Syntax

```sql
DROP VIEW IF EXISTS [Schema].[ViewName];   -- must be in its own batch
GO

CREATE VIEW [Schema].[ViewName]
AS
    SELECT ...
    FROM   ...;
GO
```

- `CREATE VIEW` must be **the first statement in a T-SQL batch** — the `GO`
  separator before it is required when preceded by any other statement.
- Column aliases defined inside the view (`AS FullName`) become the visible column
  names when callers query the view.

In [ ]:
-- ─── Section 7: Create a View ──────────────────────────────────────────────
USE DemoDB;
GO

DROP VIEW IF EXISTS [Sales].[vw_CustomerSummary];
GO

CREATE VIEW [Sales].[vw_CustomerSummary]
AS
    SELECT
        CustomerID,
        FirstName + N' ' + LastName   AS FullName,
        Email,
        CreatedDate
    FROM [Sales].[Customer];
GO

-- Query the view — behaves exactly like a table from the caller's perspective
SELECT * FROM [Sales].[vw_CustomerSummary];

### What just happened?

`Sales.vw_CustomerSummary` is now a named query stored in `DemoDB`.

The view concatenates `FirstName` and `LastName` into a single `FullName` column —
the caller sees a clean interface and never needs to know how it was computed.

Because the view holds no data, any changes to `Sales.Customer` (new rows, updates,
deletes) are immediately visible through `vw_CustomerSummary` on the very next query.
There is no refresh step, no synchronisation — it just works.

You can verify this: insert another row into `Sales.Customer` directly, then re-run
the `SELECT * FROM [Sales].[vw_CustomerSummary]` above. The new row appears instantly.

---

## Section 8 — Review and Cleanup

### What you built

Working through this notebook you created the full SQL Server object hierarchy:

```
SQL Server Instance (localhost)
└── Login:    DemoUser                              (server level — authenticates the connection)
    └── Database: DemoDB                            (isolated object container)
        ├── User:   DemoUser → db_owner             (database level — mapped to the Login)
        └── Schema: Sales                           (namespace grouping)
            ├── Table: Sales.Customer               (5 rows of structured data)
            └── View:  Sales.vw_CustomerSummary     (virtual table over Customer)
```

### Key concepts recap

| Concept | SQL Server | Snowflake equivalent |
|---------|-----------|---------------------|
| Server authentication principal | `LOGIN` (server level) | `USER` (account level) |
| Database access principal | `USER` (database level) | _(same user, role-based)_ |
| Namespace grouping | `SCHEMA` | `SCHEMA` |
| Auto-increment column | `IDENTITY(seed, increment)` | `AUTOINCREMENT` / `IDENTITY` |
| Current date function | `GETDATE()` | `CURRENT_DATE()` / `CURRENT_TIMESTAMP()` |
| Virtual table | `VIEW` | `VIEW` — identical behaviour |
| Inspect table columns | `INFORMATION_SCHEMA.COLUMNS` | `INFORMATION_SCHEMA.COLUMNS` |

### Optional Cleanup

> ⚠️ **Warning — Destructive and irreversible.**
> The SQL cell below drops **everything** created in this demo in reverse dependency order.
> All data will be permanently deleted. Only run this when you are completely done
> with the demo and no longer need `DemoDB` or the `DemoUser` login.
>
> **To run:** uncomment the SQL lines, then execute the cell.

In [ ]:
-- ─── Section 8: Cleanup — Uncomment to run ──────────────────────────────────
-- WARNING: This permanently drops all objects created in this demo.
-- Reverse dependency order: View → Table → Schema → User → Database → Login

-- -- Step 1: Drop view, table, schema, and user inside DemoDB
-- USE DemoDB;
-- GO
-- DROP VIEW  IF EXISTS [Sales].[vw_CustomerSummary];
-- DROP TABLE IF EXISTS [Sales].[Customer];
-- DROP SCHEMA IF EXISTS [Sales];
-- DROP USER  IF EXISTS [DemoUser];
-- GO

-- -- Step 2: Switch to master — you cannot drop the database you are currently using
-- USE master;
-- GO

-- -- Step 3: Drop the database (removes all remaining contained objects)
-- DROP DATABASE IF EXISTS DemoDB;
-- GO

-- -- Step 4: Drop the server-level login (no DROP LOGIN IF EXISTS in T-SQL)
-- IF EXISTS (
--     SELECT 1 FROM sys.server_principals
--     WHERE  name = N'DemoUser' AND type = 'S'
-- )
--     DROP LOGIN [DemoUser];
-- GO